# TUGAS KELOMPOK 5 — BAGIAN KAYOO
## Implementasi Deteksi Tepi: Sobel, Roberts, dan Kompas
**Mata Kuliah:** Analisis dan Desain Citra Berbasis Komputasi  
**Metode:** Deteksi Tepi Sobel | Roberts | Kompas (8 Arah)  

---

## 0. Import Library & Load Citra

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

# ── Load kedua citra ──────────────────────────────────────────────
img1_bgr = cv2.imread('image 1.jpeg')
img2_bgr = cv2.imread('image 2.jpeg')

# ── Konversi ke grayscale ─────────────────────────────────────────
gray1 = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2_bgr, cv2.COLOR_BGR2GRAY)

print(f'Citra 1 — ukuran: {gray1.shape}')
print(f'Citra 2 — ukuran: {gray2.shape}')

# ── Tampilkan citra asli ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(gray1, cmap='gray')
axes[0].set_title('Citra 1 — Grayscale', fontsize=13, fontweight='bold')
axes[0].axis('off')
axes[1].imshow(gray2, cmap='gray')
axes[1].set_title('Citra 2 — Grayscale', fontsize=13, fontweight='bold')
axes[1].axis('off')
plt.suptitle('Citra Input (Grayscale)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 1. Deteksi Tepi — SOBEL

### Cara Kerja
Operator **Sobel** menggunakan dua kernel berukuran **3×3** untuk menghitung gradien intensitas piksel arah horizontal (**Gx**) dan vertikal (**Gy**). Magnitude akhir dihitung dengan:

$$G = \sqrt{G_x^2 + G_y^2}$$

**Kernel Sobel:**
$$G_x = \begin{bmatrix} -1 & 0 & +1 \\ -2 & 0 & +2 \\ -1 & 0 & +1 \end{bmatrix}, \quad
G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ +1 & +2 & +1 \end{bmatrix}$$

Bobot **2** pada baris/kolom tengah memberikan efek smoothing ringan sehingga lebih tahan terhadap noise dibanding Roberts.

In [ ]:
# ── Implementasi Sobel ────────────────────────────────────────────
def sobel_detection(gray_img):
    gray_f = gray_img.astype(np.float64)
    Gx = cv2.Sobel(gray_f, cv2.CV_64F, 1, 0, ksize=3)   # Horizontal
    Gy = cv2.Sobel(gray_f, cv2.CV_64F, 0, 1, ksize=3)   # Vertikal
    magnitude = np.sqrt(Gx**2 + Gy**2)
    magnitude = np.clip(magnitude, 0, 255).astype(np.uint8)
    return magnitude, Gx, Gy

sobel1, Gx1, Gy1 = sobel_detection(gray1)
sobel2, Gx2, Gy2 = sobel_detection(gray2)

# ── Visualisasi ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('DETEKSI TEPI SOBEL', fontsize=16, fontweight='bold')

labels = ['Grayscale', 'Sobel Gx\n(Horizontal)', 'Sobel Gy\n(Vertikal)', 'Hasil Sobel\n(Magnitude)']
imgs1  = [gray1, np.abs(Gx1).clip(0,255).astype(np.uint8), np.abs(Gy1).clip(0,255).astype(np.uint8), sobel1]
imgs2  = [gray2, np.abs(Gx2).clip(0,255).astype(np.uint8), np.abs(Gy2).clip(0,255).astype(np.uint8), sobel2]

for j in range(4):
    axes[0,j].imshow(imgs1[j], cmap='gray')
    axes[0,j].set_title(f'CITRA 1\n{labels[j]}', fontsize=10, fontweight='bold')
    axes[0,j].axis('off')
    axes[1,j].imshow(imgs2[j], cmap='gray')
    axes[1,j].set_title(f'CITRA 2\n{labels[j]}', fontsize=10, fontweight='bold')
    axes[1,j].axis('off')

plt.tight_layout()
plt.savefig('output/hasil_sobel.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Tersimpan: output/hasil_sobel.png')

### Analisis Hasil Sobel

| Aspek | Keterangan |
|-------|------------|
| **Kernel** | 3×3, dua arah (Gx dan Gy) |
| **Ketebalan tepi** | Sedang — lebih tebal dari Roberts |
| **Ketahanan noise** | Baik — bobot tengah meredam noise |
| **Kecepatan** | Cepat |

> **Pengamatan:** Metode Sobel menghasilkan garis tepi yang jelas dan kontinu. Tepi horizontal dan vertikal terdeteksi dengan baik karena adanya dua kernel terpisah (Gx dan Gy).

---
## 2. Deteksi Tepi — ROBERTS

### Cara Kerja
Operator **Roberts Cross** menggunakan dua kernel berukuran **2×2** yang menghitung perbedaan intensitas piksel secara **diagonal**.

$$G = \sqrt{G_x^2 + G_y^2}$$

**Kernel Roberts:**
$$G_x = \begin{bmatrix} +1 & 0 \\ 0 & -1 \end{bmatrix}, \quad
G_y = \begin{bmatrix} 0 & +1 \\ -1 & 0 \end{bmatrix}$$

Ukuran kernel yang sangat kecil (2×2) membuat Roberts sangat sensitif terhadap perubahan intensitas tajam, namun juga rentan terhadap noise.

In [ ]:
# ── Implementasi Roberts ──────────────────────────────────────────
def roberts_detection(gray_img):
    gray_f = gray_img.astype(np.float64)
    kernel_x = np.array([[ 1,  0],
                          [ 0, -1]], dtype=np.float64)
    kernel_y = np.array([[ 0,  1],
                          [-1,  0]], dtype=np.float64)
    Gx = cv2.filter2D(gray_f, cv2.CV_64F, kernel_x)
    Gy = cv2.filter2D(gray_f, cv2.CV_64F, kernel_y)
    magnitude = np.sqrt(Gx**2 + Gy**2)
    magnitude = np.clip(magnitude, 0, 255).astype(np.uint8)
    return magnitude, Gx, Gy

roberts1, Rx1, Ry1 = roberts_detection(gray1)
roberts2, Rx2, Ry2 = roberts_detection(gray2)

# ── Visualisasi ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('DETEKSI TEPI ROBERTS', fontsize=16, fontweight='bold')

labels = ['Grayscale', 'Roberts Gx\n(Diagonal 45°)', 'Roberts Gy\n(Diagonal 135°)', 'Hasil Roberts\n(Magnitude)']
imgs1  = [gray1, np.abs(Rx1).clip(0,255).astype(np.uint8), np.abs(Ry1).clip(0,255).astype(np.uint8), roberts1]
imgs2  = [gray2, np.abs(Rx2).clip(0,255).astype(np.uint8), np.abs(Ry2).clip(0,255).astype(np.uint8), roberts2]

for j in range(4):
    axes[0,j].imshow(imgs1[j], cmap='gray')
    axes[0,j].set_title(f'CITRA 1\n{labels[j]}', fontsize=10, fontweight='bold')
    axes[0,j].axis('off')
    axes[1,j].imshow(imgs2[j], cmap='gray')
    axes[1,j].set_title(f'CITRA 2\n{labels[j]}', fontsize=10, fontweight='bold')
    axes[1,j].axis('off')

plt.tight_layout()
plt.savefig('output/hasil_roberts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Tersimpan: output/hasil_roberts.png')

### Analisis Hasil Roberts

| Aspek | Keterangan |
|-------|------------|
| **Kernel** | 2×2, dua arah diagonal |
| **Ketebalan tepi** | Tipis dan tajam |
| **Ketahanan noise** | Rendah — sangat sensitif noise |
| **Kecepatan** | Paling cepat (kernel terkecil) |

> **Pengamatan:** Metode Roberts menghasilkan tepi yang lebih tipis dibanding Sobel. Pada area dengan noise, Roberts cenderung menghasilkan garis tepi yang terputus-putus. Deteksi tepi diagonal (45° dan 135°) lebih menonjol dibanding arah lainnya.

---
## 3. Deteksi Tepi — KOMPAS (Kirsch Compass)

### Cara Kerja
Operator **Kompas** (Kirsch Compass) menggunakan **8 kernel 3×3** yang merepresentasikan 8 arah mata angin:
**N, NE, E, SE, S, SW, W, NW**.

Citra dikonvolusikan dengan setiap kernel, lalu nilai tepi akhir diambil dari **respons maksimum** di antara semua 8 arah:

$$G = \max(|R_N|, |R_{NE}|, |R_E|, \ldots, |R_{NW}|)$$

Pendekatan ini memungkinkan deteksi tepi dari **berbagai orientasi** sekaligus, sehingga menghasilkan tepi yang lebih lengkap dan detail.

In [ ]:
# ── Implementasi Kompas (Kirsch) ──────────────────────────────────
def compass_detection(gray_img):
    gray_f = gray_img.astype(np.float64)

    kernels = {
        'N  (Utara)':      np.array([[ 5,  5,  5], [-3,  0, -3], [-3, -3, -3]], dtype=np.float64),
        'NE (Timur Laut)': np.array([[-3,  5,  5], [-3,  0,  5], [-3, -3, -3]], dtype=np.float64),
        'E  (Timur)':      np.array([[-3, -3,  5], [-3,  0,  5], [-3, -3,  5]], dtype=np.float64),
        'SE (Tenggara)':   np.array([[-3, -3, -3], [-3,  0,  5], [-3,  5,  5]], dtype=np.float64),
        'S  (Selatan)':    np.array([[-3, -3, -3], [-3,  0, -3], [ 5,  5,  5]], dtype=np.float64),
        'SW (Barat Daya)': np.array([[-3, -3, -3], [ 5,  0, -3], [ 5,  5, -3]], dtype=np.float64),
        'W  (Barat)':      np.array([[ 5, -3, -3], [ 5,  0, -3], [ 5, -3, -3]], dtype=np.float64),
        'NW (Barat Laut)': np.array([[ 5,  5, -3], [ 5,  0, -3], [-3, -3, -3]], dtype=np.float64),
    }

    responses = {}
    for direction, kernel in kernels.items():
        resp = cv2.filter2D(gray_f, cv2.CV_64F, kernel)
        responses[direction] = np.abs(resp)

    result = np.maximum.reduce(list(responses.values()))
    result = np.clip(result, 0, 255).astype(np.uint8)
    return result, responses

compass1, resp1 = compass_detection(gray1)
compass2, resp2 = compass_detection(gray2)
print('✅ Kompas selesai diproses')

In [ ]:
# ── Visualisasi 8 arah — Citra 1 ─────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(25, 10))
fig.suptitle('DETEKSI TEPI KOMPAS — CITRA 1 (8 Arah Mata Angin)', fontsize=14, fontweight='bold')

axes[0,0].imshow(gray1, cmap='gray')
axes[0,0].set_title('Citra 1 Asli\n(Grayscale)', fontweight='bold')
axes[0,0].axis('off')

for idx, (direction, resp) in enumerate(resp1.items()):
    row, col = (idx+1) // 5, (idx+1) % 5
    axes[row,col].imshow(np.clip(resp, 0, 255).astype(np.uint8), cmap='gray')
    axes[row,col].set_title(f'Arah {direction}', fontsize=9)
    axes[row,col].axis('off')

axes[1,4].imshow(compass1, cmap='gray')
axes[1,4].set_title('HASIL AKHIR\n(Maks. 8 Arah)', fontweight='bold', color='red')
axes[1,4].axis('off')

plt.tight_layout()
plt.savefig('output/hasil_kompas_citra1.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Tersimpan: output/hasil_kompas_citra1.png')

In [ ]:
# ── Visualisasi 8 arah — Citra 2 ─────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(25, 10))
fig.suptitle('DETEKSI TEPI KOMPAS — CITRA 2 (8 Arah Mata Angin)', fontsize=14, fontweight='bold')

axes[0,0].imshow(gray2, cmap='gray')
axes[0,0].set_title('Citra 2 Asli\n(Grayscale)', fontweight='bold')
axes[0,0].axis('off')

for idx, (direction, resp) in enumerate(resp2.items()):
    row, col = (idx+1) // 5, (idx+1) % 5
    axes[row,col].imshow(np.clip(resp, 0, 255).astype(np.uint8), cmap='gray')
    axes[row,col].set_title(f'Arah {direction}', fontsize=9)
    axes[row,col].axis('off')

axes[1,4].imshow(compass2, cmap='gray')
axes[1,4].set_title('HASIL AKHIR\n(Maks. 8 Arah)', fontweight='bold', color='red')
axes[1,4].axis('off')

plt.tight_layout()
plt.savefig('output/hasil_kompas_citra2.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Tersimpan: output/hasil_kompas_citra2.png')

### Analisis Hasil Kompas

| Aspek | Keterangan |
|-------|------------|
| **Kernel** | 3×3, sebanyak 8 arah mata angin |
| **Ketebalan tepi** | Tebal — paling detail |
| **Ketahanan noise** | Sedang |
| **Kecepatan** | Paling lambat (8× konvolusi) |

> **Pengamatan:** Operator Kompas menghasilkan tepi yang paling lengkap karena mendeteksi gradien dari 8 arah sekaligus. Tepi dari berbagai orientasi (tidak hanya horizontal/vertikal) berhasil terdeteksi dengan baik. Kompleksitas komputasi lebih tinggi karena melakukan 8 kali konvolusi per piksel.

---
## 4. Perbandingan Akhir — Sobel vs Roberts vs Kompas

In [ ]:
# ── Perbandingan semua metode ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
fig.suptitle('PERBANDINGAN METODE DETEKSI TEPI — BAGIAN KAYOO\nSobel | Roberts | Kompas',
             fontsize=15, fontweight='bold')

methods = ['Grayscale (Asli)', 'Sobel', 'Roberts', 'Kompas']
colors  = ['black', '#1a6fb5', '#c0392b', '#27ae60']
imgs1   = [gray1, sobel1, roberts1, compass1]
imgs2   = [gray2, sobel2, roberts2, compass2]

for j in range(4):
    axes[0,j].imshow(imgs1[j], cmap='gray')
    axes[0,j].set_title(f'CITRA 1\n{methods[j]}', fontsize=12,
                        fontweight='bold', color=colors[j])
    axes[0,j].axis('off')
    axes[1,j].imshow(imgs2[j], cmap='gray')
    axes[1,j].set_title(f'CITRA 2\n{methods[j]}', fontsize=12,
                        fontweight='bold', color=colors[j])
    axes[1,j].axis('off')

plt.tight_layout()
plt.savefig('output/perbandingan_kayoo.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Tersimpan: output/perbandingan_kayoo.png')

In [ ]:
# ── Analisis Statistik Kuantitatif ────────────────────────────────
threshold = 50  # Ambang batas piksel dianggap tepi

print('=' * 65)
print('ANALISIS KUANTITATIF — BAGIAN KAYOO')
print('=' * 65)
print(f'{"Metode":<12} {"Citra":<10} {"Rata-Rata":>10} {"Std Dev":>10} {"Piksel Tepi":>13}')
print('-' * 65)

data = [
    ('Sobel',   sobel1,   sobel2),
    ('Roberts', roberts1, roberts2),
    ('Kompas',  compass1, compass2),
]

for method, r1, r2 in data:
    for label, r in [('Citra 1', r1), ('Citra 2', r2)]:
        mean  = r.mean()
        std   = r.std()
        edges = np.sum(r > threshold)
        print(f'{method:<12} {label:<10} {mean:>10.2f} {std:>10.2f} {edges:>13}')
    print()

print('=' * 65)
print('KESIMPULAN:')
print('  Sobel   → Tepi jelas, tahan noise, kernel 3×3 berbobot')
print('  Roberts → Tepi tipis & tajam, sensitif noise, kernel 2×2')
print('  Kompas  → Tepi paling detail, mendeteksi 8 arah sekaligus')
print('=' * 65)